# How to Run

**Delete this cell before final submission — it's for your reference only.**

### What this notebook does
This is the most important notebook to run first on your desktop. The RTX 5080 uses NVIDIA's new Blackwell architecture (`sm_120`), which requires a recent PyTorch build (≥ 2.7) and CUDA 12.8 wheels. Older PyTorch versions install silently and then fall back to CPU at runtime, which is the single most common reason students lose a day of training time on new GPUs. This notebook installs the right versions and verifies everything actually works on your GPU.

### Where to run it
**On your desktop (the RTX 5080 machine), in a Python venv.** Not on Kaggle.

### Setup steps (do these in a terminal first, not in the notebook)

1. **Install Python 3.11 or 3.12** if you don't already have it. Get it from [python.org](https://www.python.org/downloads/windows/). When the installer asks, tick *Add Python to PATH*.

2. **Open PowerShell** and `cd` into a project folder of your choice:
   ```powershell
   cd C:\Users\<you>\Documents\deepfake_project
   ```

3. **Create and activate a venv:**
   ```powershell
   python -m venv .venv
   .\.venv\Scripts\Activate.ps1
   ```
   You should see `(.venv)` at the start of your prompt.

4. **Upgrade pip and install JupyterLab:**
   ```powershell
   python -m pip install --upgrade pip
   pip install jupyterlab
   ```

5. **Install PyTorch with CUDA 12.8 support (the Blackwell-compatible wheels):**
   ```powershell
   pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
   ```
   This is the critical line. Do not omit `--index-url` — without it, pip installs CPU-only PyTorch.

6. **Install the rest of the project dependencies:**
   ```powershell
   pip install decord facenet-pytorch timm tqdm matplotlib pandas scikit-learn opencv-python pillow
   ```

7. **Launch JupyterLab:**
   ```powershell
   jupyter lab
   ```
   A browser tab opens. Open this notebook from the file browser.

8. **Run all cells.** Every check should print a green success message. If any check fails, the cell tells you exactly what to fix.

### Expected runtime
About 1 minute (mostly downloading a small test tensor to GPU).

### If something fails
Do not skip past failed checks. The whole training pipeline depends on PyTorch correctly recognizing your GPU. The most common fix is reinstalling PyTorch with the right `--index-url`.


# 00b — Environment Setup Verification

We verify five things in order. If any one fails, the next ones don't matter.

1. Python version
2. PyTorch version (must be ≥ 2.7)
3. CUDA available
4. GPU detected and named
5. **Compute capability includes sm_120** (the critical Blackwell check)
6. End-to-end GPU operation works
7. All other dependencies importable

## 1. Python version

In [1]:
import sys

python_major, python_minor = sys.version_info.major, sys.version_info.minor
print(f'Python version: {sys.version}')

assert python_major == 3 and python_minor >= 10, (
    f'Need Python 3.10 or newer; you have {python_major}.{python_minor}.'
)
print('✓ Python version OK')

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
✓ Python version OK


## 2. PyTorch version

PyTorch 2.7 was the first stable release with native Blackwell (`sm_120`) support.

In [2]:
#!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [3]:
import torch
from packaging import version

torch_version_string = torch.__version__
print(f"PyTorch version: {torch_version_string}")

# packaging handles '2.7.0rc1' and '2.7.0+cu128' flawlessly
current_version = version.parse(torch_version_string)
minimum_version = version.parse("2.7.0")

if current_version < minimum_version:
    raise RuntimeError(
        f"PyTorch {torch_version_string} is too old for RTX 5080 (Blackwell). "
        f"You need ≥ 2.7. Reinstall with:\n"
        f"  pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128"
    )

if not torch.cuda.is_available():
    print("⚠ Warning: PyTorch is running in CPU-only mode.")
else:
    print(f"✓ PyTorch version OK (Running on: {torch.cuda.get_device_name(0)})")


PyTorch version: 2.11.0+cu128
✓ PyTorch version OK (Running on: NVIDIA GeForce RTX 5080)


## 3. CUDA available

In [4]:
cuda_is_available = torch.cuda.is_available()
print(f'CUDA available: {cuda_is_available}')

if not cuda_is_available:
    raise RuntimeError(
        'CUDA is not available. Possible causes:\n'
        '  1. PyTorch was installed without CUDA support (use --index-url cu128).\n'
        '  2. NVIDIA driver too old. Update to the latest Game-Ready driver.\n'
        '  3. GPU not detected by the OS. Check Device Manager.'
    )

print(f'CUDA runtime version PyTorch was built with: {torch.version.cuda}')
print('✓ CUDA available')

CUDA available: True
CUDA runtime version PyTorch was built with: 12.8
✓ CUDA available


## 4. GPU detected and named

In [5]:
gpu_count = torch.cuda.device_count()
print(f'Number of CUDA GPUs visible: {gpu_count}')

for gpu_index in range(gpu_count):
    gpu_name = torch.cuda.get_device_name(gpu_index)
    gpu_total_memory_gb = torch.cuda.get_device_properties(gpu_index).total_memory / 1e9
    print(f'  GPU {gpu_index}: {gpu_name}  ({gpu_total_memory_gb:.1f} GB)')

assert gpu_count >= 1, 'No GPU detected.'
print('✓ GPU detected')

Number of CUDA GPUs visible: 1
  GPU 0: NVIDIA GeForce RTX 5080  (17.1 GB)
✓ GPU detected


## 5. Compute capability check (the critical one)

Your RTX 5080 has compute capability **12.0** (Blackwell, `sm_120`). PyTorch was compiled with support for a set of architectures — this list **must include 12.0** or PyTorch silently falls back to CPU at runtime.

In [6]:
# Your GPU's actual compute capability:
device_properties = torch.cuda.get_device_properties(0)
device_compute_capability = (device_properties.major, device_properties.minor)
print(f'Your GPU compute capability: sm_{device_properties.major}{device_properties.minor}')

# Architectures PyTorch was actually compiled to support:
supported_architecture_list = torch.cuda.get_arch_list()
print(f'PyTorch was compiled to support: {supported_architecture_list}')

device_arch_string = f'sm_{device_properties.major}{device_properties.minor}'
is_supported_natively = any(device_arch_string in arch for arch in supported_architecture_list)

if not is_supported_natively:
    print(f'\n⚠ CRITICAL: PyTorch does not natively support {device_arch_string}.')
    print('  Your training will silently fall back to CPU and be ~100x slower.')
    print('  Fix by reinstalling PyTorch with the CUDA 12.8 index URL:')
    print('    pip install --upgrade --force-reinstall torch torchvision torchaudio \\')
    print('      --index-url https://download.pytorch.org/whl/cu128')
    raise RuntimeError('PyTorch was not built for your GPU architecture.')

print(f'✓ {device_arch_string} is natively supported')

Your GPU compute capability: sm_120
PyTorch was compiled to support: ['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
✓ sm_120 is natively supported


## 6. End-to-end GPU operation

Do a small matrix multiplication on GPU. This catches any cases where the previous checks passed but actual GPU execution fails.

In [7]:
import time

device = torch.device('cuda:0')

# Two largish random matrices.
matrix_a = torch.randn(4096, 4096, device=device, dtype=torch.float16)
matrix_b = torch.randn(4096, 4096, device=device, dtype=torch.float16)

# Warm up (first matmul includes kernel compilation time).
_ = matrix_a @ matrix_b
torch.cuda.synchronize()

# Timed run.
start_time = time.perf_counter()
result_matrix = matrix_a @ matrix_b
torch.cuda.synchronize()
elapsed_seconds = time.perf_counter() - start_time

print(f'4096×4096 fp16 matmul ran in {elapsed_seconds*1000:.2f} ms')
print(f'Result tensor on device: {result_matrix.device}')

# On RTX 5080 this should be roughly 2–10 ms. If it's > 500 ms, you're on CPU.
if elapsed_seconds > 0.5:
    print('⚠ Warning: matmul was unexpectedly slow. You may be running on CPU.')
else:
    print('✓ GPU compute working at expected speed')

4096×4096 fp16 matmul ran in 2.11 ms
Result tensor on device: cuda:0
✓ GPU compute working at expected speed


## 7. Other dependencies

Confirm every other package the project needs is importable.

In [8]:
required_packages = [
    'torchvision',
    'decord',
    'facenet_pytorch',
    'timm',
    'tqdm',
    'matplotlib',
    'pandas',
    'sklearn',         # scikit-learn imports as 'sklearn'
    'cv2',             # opencv-python imports as 'cv2'
    'PIL',             # pillow imports as 'PIL'
    'numpy',
]

missing_packages = []
for package_name in required_packages:
    try:
        __import__(package_name)
        print(f'  ✓ {package_name}')
    except ImportError as e:
        print(f'  ✗ {package_name}  ({e})')
        missing_packages.append(package_name)

if missing_packages:
    print(f'\n⚠ Install missing packages with:')
    print(f'  pip install {" ".join(missing_packages)}')
else:
    print('\n✓ All dependencies present')

  ✓ torchvision
  ✓ decord


C:\x-ion\TEST\xion\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓ facenet_pytorch
  ✓ timm
  ✓ tqdm
  ✓ matplotlib
  ✓ pandas
  ✓ sklearn
  ✓ cv2
  ✓ PIL
  ✓ numpy

✓ All dependencies present


## 8. Summary

If all checks above passed:
- Python + PyTorch + CUDA + Blackwell support are all working.
- All dependencies are installed.
- You can move on to `00_dataset_exploration.ipynb` and then `01_preprocessing.ipynb`.

If anything failed, fix it now. Every later notebook assumes this environment works.